In [2]:
"""
generate_attack_trees_interactive.py

For each tactic, generates:
  - attack_tree_tactic_XX_<Tactic>.dot   (Graphviz source)
  - attack_tree_tactic_XX_<Tactic>.svg   (Graphviz-rendered SVG)
  - attack_tree_tactic_XX_<Tactic>.html  (interactive: click node → metadata panel)
  - attack_tree_tactic_XX_<Tactic>.json  (node/edge metadata)

Both bugs from the original generator are fixed:
  Bug 1 — dedup key now includes the CWE name so two CWEs sharing a technique
           each build their own subtree.
  Bug 2 — multiple AND gates under a single CWE are joined by an OR gate before
           wiring to the CWE node.

Dependencies:
    pip install pandas openpyxl graphviz
    System: Graphviz `dot` binary must be on PATH.
"""

import os
import re
import json
import html as html_lib
import pandas as pd
import xml.etree.ElementTree as ET
from graphviz import Digraph

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────

DATASET_PATH = "device_compromise_dataset.xlsx"
OUTPUT_DIR   = "attack_tree_images"

MITRE_TACTIC_ORDER = [
    "Reconnaissance", "Resource Development", "Initial Access", "Execution",
    "Persistence", "Privilege Escalation", "Defense Evasion", "Credential Access",
    "Discovery", "Lateral Movement", "Collection", "Command and Control",
    "Exfiltration", "Impact",
]

# Node fill colours (kept identical to original)
COLORS = {
    "root":    "#FFD700",
    "tactic":  "#98FB98",
    "cwe":     "#ADD8E6",
    "tech":    "#90EE90",
    "subtech": "#FFFACD",
    "capec":   "#FFB6C1",
    "gate":    "#FFCCCB",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Load data
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_excel(DATASET_PATH, index_col=False)
df["tactics"] = df["tactics"].astype(str).str.strip()
df = df.fillna("")

unique_tactics   = [t for t in df["tactics"].unique() if t]
ordered_tactics  = [t for t in MITRE_TACTIC_ORDER if t in unique_tactics]
# append any tactics not in MITRE list
for t in unique_tactics:
    if t not in ordered_tactics:
        ordered_tactics.append(t)

print(f"Found {len(ordered_tactics)} tactics: {ordered_tactics}")


# ─────────────────────────────────────────────────────────────────────────────
# Metadata helpers
# ─────────────────────────────────────────────────────────────────────────────

def _norm(s):
    return re.sub(r"[^0-9a-z]", "", str(s).lower())


def extract_node_metadata(df_subset: pd.DataFrame, node_type: str) -> dict:
    """Aggregate unique non-empty column values for a node."""
    cols_map = {_norm(c): c for c in df_subset.columns}

    key_groups = {
        "cwe": [
            "CVE ID", "cwe_Name", "cwe_Abstraction", "cwe_Modes_Of_Introduction",
            "cwe_Likelihood_Of_Exploit", "cwe_Common_Consequences",
            "cwe_Potential_Mitigations", "attackVector", "baseScore", "baseSeverity",
            "attackComplexity", "privilegesRequired", "userInteraction",
            "confidentialityImpact", "availabilityImpact", "exploitabilityScore",
        ],
        "capec": [
            "CVE ID", "capec_Name", "capec_Likelihood_Of_Attack",
            "capec_Typical_Severity", "capec_Skills_Required",
            "capec_Prerequisites", "capec_Mitigations", "capec_Consequences",
        ],
        "tech": [
            "CVE ID", "technique_name", "subtechnique_name", "tactics",
            "tech_description", "detection", "platforms", "data sources",
            "defenses bypassed", "permissions required",
        ],
        "subtech": [
            "subtechnique_name", "technique_name", "tactics",
            "detection", "platforms", "data sources",
        ],
    }
    keys = key_groups.get(node_type, list(df_subset.columns))

    meta = {}
    for k in keys:
        col = cols_map.get(_norm(k))
        if col and col in df_subset.columns:
            vals = [
                v for v in df_subset[col].astype(str).str.strip().tolist()
                if v and v.lower() != "nan"
            ]
            uniq = list(dict.fromkeys(vals))   # preserve order, deduplicate
            if uniq:
                meta[k] = " | ".join(uniq)
    return meta


def meta_to_html_table(meta: dict) -> str:
    """Render a metadata dict as an HTML table string."""
    if not meta:
        return "<div class='meta-empty'>No metadata available.</div>"

    preferred = [
        "cwe_Name", "capec_Name", "technique_name", "subtechnique_name",
        "CVE ID", "baseScore", "baseSeverity", "attackVector",
    ]
    ordered_keys = [k for k in preferred if k in meta]
    ordered_keys += [k for k in meta if k not in ordered_keys]

    rows = []
    for k in ordered_keys:
        k_e = html_lib.escape(str(k))
        v_e = html_lib.escape(str(meta[k]))
        rows.append(
            f"<tr>"
            f"<th>{k_e}</th>"
            f"<td>{v_e}</td>"
            f"</tr>"
        )
    return "<table class='meta-table'>" + "".join(rows) + "</table>"


# ─────────────────────────────────────────────────────────────────────────────
# Graph-building helpers
# ─────────────────────────────────────────────────────────────────────────────

def _add_node(dot, nid, label, ntype, nodes_list, metadata=None):
    shape  = "note" if ntype == "cwe" else "ellipse" if ntype == "gate" else "box"
    color  = COLORS.get(ntype, "#FFFFFF")
    dot.node(nid, label=label, shape=shape, fillcolor=color,
             style="filled", fontname="Helvetica")
    entry = {"id": nid, "label": label, "type": ntype}
    if metadata:
        entry["metadata"] = metadata
    nodes_list.append(entry)


def _add_edge(dot, src, dst, edges_list):
    dot.edge(src, dst)
    edges_list.append({"from": src, "to": dst})


def binary_or(dot, parent_id, child_ids, prefix, counter, nodes_list, edges_list):
    """
    Connect parent_id to child_ids through a balanced binary OR-gate tree.
    counter is a one-element list used as a mutable int.
    """
    if not child_ids:
        return
    if len(child_ids) == 1:
        _add_edge(dot, parent_id, child_ids[0], edges_list)
        return

    def _build(children):
        if len(children) == 1:
            return children[0]
        gid = f"{prefix}_or_{counter[0]}"
        counter[0] += 1
        _add_node(dot, gid, "OR", "gate", nodes_list)
        mid = len(children) // 2
        left  = _build(children[:mid])
        right = _build(children[mid:])
        _add_edge(dot, gid, left,  edges_list)
        _add_edge(dot, gid, right, edges_list)
        return gid

    gate_root = _build(child_ids)
    _add_edge(dot, parent_id, gate_root, edges_list)


# ─────────────────────────────────────────────────────────────────────────────
# Per-tactic graph builder  (both bugs fixed)
# ─────────────────────────────────────────────────────────────────────────────

def build_tactic_graph(tactic: str, tactic_index: int):
    """
    Returns (dot, nodes_list, edges_list, node_html_map)
    node_html_map: {node_id: html_table_string}
    """
    dot = Digraph()
    dot.attr("graph", rankdir="TB", splines="ortho",
             nodesep="0.5", ranksep="0.8")
    dot.attr("node", fontname="Helvetica", fontsize="12")

    nodes_list  = []
    edges_list  = []
    node_html   = {}   # node_id -> HTML metadata table
    counter     = [0]  # shared mutable counter for gate IDs

    prefix = f"t{tactic_index}"

    # ── root + tactic ────────────────────────────────────────────────────────
    root_id   = "root"
    tactic_id = f"{prefix}_tactic"

    _add_node(dot, root_id,   "Compromise Device", "root",   nodes_list)
    _add_node(dot, tactic_id, tactic,              "tactic", nodes_list,
              metadata={"tactic": tactic})
    _add_edge(dot, root_id, tactic_id, edges_list)
    node_html[root_id]   = meta_to_html_table({"label": "Compromise Device"})
    node_html[tactic_id] = meta_to_html_table({"tactic": tactic})

    # ── get CWEs for this tactic ─────────────────────────────────────────────
    tactic_df = df[df["tactics"] == tactic]
    cwe_name_set = {
        str(c).strip().lower()
        for c in tactic_df["cwe_Name"].dropna().unique()
        if str(c).strip()
    }
    tactic_weaknesses = [
        str(c).strip()
        for c in tactic_df["cwe_Name"].dropna().unique()
        if str(c).strip()
    ]

    cwe_ids = []

    for ci, cwe in enumerate(tactic_weaknesses):
        cwe_id     = f"{prefix}_cwe_{ci}"
        cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
        cwe_meta   = extract_node_metadata(cwe_subset, "cwe")

        _add_node(dot, cwe_id, cwe, "cwe", nodes_list, metadata=cwe_meta)
        node_html[cwe_id] = meta_to_html_table(cwe_meta)
        cwe_ids.append(cwe_id)

        # ── techniques under this CWE ─────────────────────────────────────
        techniques = [
            t for t in cwe_subset["technique_name"].dropna().unique()
            if str(t).strip()
        ]

        and_gate_ids = []   # one AND gate per (technique/subtech + capec) pair

        # FIX 1: dedup key includes cwe so same technique under different CWEs
        #         each gets its own subtree
        seen_tech_keys = set()

        for technique in techniques:
            tech_key = f"{tactic}:{cwe}:{technique}"
            if tech_key in seen_tech_keys:
                continue
            seen_tech_keys.add(tech_key)

            tech_df = cwe_subset[cwe_subset["technique_name"] == technique]

            subtechs = [
                s for s in tech_df["subtechnique_name"].dropna().unique()
                if str(s).strip() and str(s).strip().lower() not in cwe_name_set
            ]
            capec_names = [
                c for c in tech_df["capec_Name"].dropna().unique()
                if str(c).strip()
            ]
            capec_name = capec_names[0] if capec_names else "CAPEC ?"

            if subtechs:
                for subtech in subtechs:
                    n = counter[0]; counter[0] += 1

                    # subtechnique node
                    sub_id  = f"{prefix}_sub_{n}"
                    sub_rows = tech_df[tech_df["subtechnique_name"] == subtech]
                    sub_meta = extract_node_metadata(sub_rows, "subtech")
                    _add_node(dot, sub_id, subtech, "subtech", nodes_list, metadata=sub_meta)
                    node_html[sub_id] = meta_to_html_table(sub_meta)

                    # capec node
                    cap_id   = f"{prefix}_cap_{n}"
                    cap_rows = tech_df[tech_df["capec_Name"] == capec_name] \
                               if capec_name != "CAPEC ?" else tech_df
                    cap_meta = extract_node_metadata(cap_rows, "capec")
                    _add_node(dot, cap_id, capec_name, "capec", nodes_list, metadata=cap_meta)
                    node_html[cap_id] = meta_to_html_table(cap_meta)

                    # AND gate
                    and_id = f"{prefix}_and_{n}"
                    _add_node(dot, and_id, "AND", "gate", nodes_list)
                    _add_edge(dot, and_id, sub_id, edges_list)
                    _add_edge(dot, and_id, cap_id, edges_list)
                    and_gate_ids.append(and_id)

            else:
                n = counter[0]; counter[0] += 1

                # technique node
                tech_id   = f"{prefix}_tech_{n}"
                tech_meta = extract_node_metadata(tech_df, "tech")
                _add_node(dot, tech_id, technique, "tech", nodes_list, metadata=tech_meta)
                node_html[tech_id] = meta_to_html_table(tech_meta)

                # capec node
                cap_id   = f"{prefix}_cap_{n}"
                cap_rows = tech_df[tech_df["capec_Name"] == capec_name] \
                           if capec_name != "CAPEC ?" else tech_df
                cap_meta = extract_node_metadata(cap_rows, "capec")
                _add_node(dot, cap_id, capec_name, "capec", nodes_list, metadata=cap_meta)
                node_html[cap_id] = meta_to_html_table(cap_meta)

                # AND gate
                and_id = f"{prefix}_and_{n}"
                _add_node(dot, and_id, "AND", "gate", nodes_list)
                _add_edge(dot, and_id, tech_id, edges_list)
                _add_edge(dot, and_id, cap_id,  edges_list)
                and_gate_ids.append(and_id)

        # FIX 2: join AND gates via OR before connecting to CWE
        binary_or(dot, cwe_id, and_gate_ids, prefix + f"_cwe{ci}",
                  counter, nodes_list, edges_list)

    # connect tactic → CWEs via OR
    binary_or(dot, tactic_id, cwe_ids, prefix + "_tac",
              counter, nodes_list, edges_list)

    return dot, nodes_list, edges_list, node_html


# ─────────────────────────────────────────────────────────────────────────────
# SVG post-processing: inject data-node-id on every <g> that has a <title>
# ─────────────────────────────────────────────────────────────────────────────

def inject_node_ids(svg_path: str) -> str:
    """
    Parse the Graphviz SVG, set data-node-id on each node <g>,
    replace <title> text with just the node label (no internal IDs shown on hover),
    and return the modified SVG as a string.
    """
    NS = "http://www.w3.org/2000/svg"
    ET.register_namespace("", NS)
    ET.register_namespace("xlink", "http://www.w3.org/1999/xlink")

    tree = ET.parse(svg_path)
    root = tree.getroot()

    for g in root.findall(f".//{{{NS}}}g"):
        title_el = g.find(f"{{{NS}}}title")
        if title_el is None or not title_el.text:
            continue
        node_id = title_el.text.strip()
        g.set("data-node-id", node_id)

        # Replace <title> with the text label of the node for nicer hover tooltip
        text_el = g.find(f".//{{{NS}}}text")
        if text_el is not None and text_el.text:
            title_el.text = text_el.text.strip()

    # Serialise back to string
    svg_str = ET.tostring(root, encoding="unicode", xml_declaration=False)
    return svg_str


# ─────────────────────────────────────────────────────────────────────────────
# HTML generator
# ─────────────────────────────────────────────────────────────────────────────

_HTML_TEMPLATE = """\
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>Attack Tree — {tactic}</title>
  <style>
    *, *::before, *::after {{ box-sizing: border-box; }}
    html, body {{ height: 100%; margin: 0; padding: 0; background: #f4f4f4;
                  font-family: Arial, Helvetica, sans-serif; font-size: 15px; }}

    /* ── Two-column app layout ── */
    .app {{ display: flex; height: 100vh; gap: 0; overflow: hidden; }}

    /* ── Left: graph canvas ── */
    .graph-col {{
      flex: 1 1 auto;
      display: flex;
      flex-direction: column;
      overflow: hidden;
      background: #fff;
      border-right: 2px solid #d0d0d0;
    }}
    .graph-header {{
      padding: 10px 16px 8px;
      border-bottom: 1px solid #e0e0e0;
      background: #fff;
    }}
    .graph-header h1 {{ margin: 0 0 6px; font-size: 20px; }}
    .legend {{ display: flex; flex-wrap: wrap; gap: 6px; margin-top: 4px; }}
    .legend-item {{
      padding: 3px 10px; border-radius: 4px;
      font-size: 13px; border: 1px solid rgba(0,0,0,.12);
    }}
    .svg-scroll {{
      flex: 1 1 auto;
      overflow: auto;
      padding: 12px;
      background: #fafafa;
    }}
    .svg-scroll svg {{
      /* keep native Graphviz size; user can scroll / zoom */
      width: auto;
      height: auto;
      display: block;
      cursor: default;
    }}
    /* highlight hovered node groups */
    .svg-scroll [data-node-id] {{ cursor: pointer; }}
    .svg-scroll [data-node-id]:hover ellipse,
    .svg-scroll [data-node-id]:hover polygon,
    .svg-scroll [data-node-id]:hover path {{
      filter: brightness(0.88);
    }}
    .selected-node ellipse,
    .selected-node polygon,
    .selected-node path {{
      stroke: #1a6fc4 !important;
      stroke-width: 3px !important;
    }}
    .graph-footer {{
      padding: 6px 16px;
      font-size: 13px;
      color: #666;
      border-top: 1px solid #e0e0e0;
      background: #fff;
    }}

    /* ── Right: metadata panel ── */
    .panel-col {{
      flex: 0 0 400px;
      display: flex;
      flex-direction: column;
      background: #fff;
      overflow: hidden;
      transition: flex-basis .2s;
    }}
    .panel-col.hidden {{ flex: 0 0 0; overflow: hidden; }}
    .panel-header {{
      display: flex;
      align-items: center;
      justify-content: space-between;
      padding: 10px 14px;
      border-bottom: 1px solid #e0e0e0;
      background: #f8f8f8;
    }}
    .panel-header h2 {{ margin: 0; font-size: 16px; }}
    .close-btn {{
      border: none; background: transparent;
      font-size: 22px; cursor: pointer; line-height: 1; padding: 0 4px;
      color: #555;
    }}
    .close-btn:hover {{ color: #000; }}
    .panel-body {{
      flex: 1 1 auto;
      overflow-y: auto;
      padding: 14px;
    }}
    .panel-placeholder {{
      color: #888; font-style: italic; margin-top: 20px; text-align: center;
    }}
    .node-label-heading {{
      font-size: 17px; font-weight: 700;
      margin: 0 0 4px;
      word-break: break-word;
    }}
    .node-type-badge {{
      display: inline-block;
      padding: 2px 8px; border-radius: 20px;
      font-size: 12px; font-weight: 600;
      margin-bottom: 10px;
    }}

    /* Metadata table */
    .meta-table {{
      width: 100%; border-collapse: collapse;
      margin-top: 8px; font-size: 14px;
    }}
    .meta-table th {{
      text-align: left; padding: 6px 8px;
      background: #f3f3f3; border-bottom: 1px solid #e0e0e0;
      font-weight: 600; width: 36%; vertical-align: top;
      word-break: break-word;
    }}
    .meta-table td {{
      padding: 6px 8px; border-bottom: 1px solid #eee;
      vertical-align: top; word-break: break-word;
    }}
    .meta-empty {{ color: #888; font-style: italic; }}

    /* Badge colours by node type */
    .badge-root    {{ background: #FFD700; color: #5a4200; }}
    .badge-tactic  {{ background: #98FB98; color: #1a5c1a; }}
    .badge-cwe     {{ background: #ADD8E6; color: #1a3a5c; }}
    .badge-tech    {{ background: #90EE90; color: #1a5c1a; }}
    .badge-subtech {{ background: #FFFACD; color: #5c5000; }}
    .badge-capec   {{ background: #FFB6C1; color: #6b0a1e; }}
    .badge-gate    {{ background: #FFCCCB; color: #7a1a00; }}
  </style>
</head>
<body>
<div class="app">

  <!-- ── Left column: SVG graph ── -->
  <div class="graph-col">
    <div class="graph-header">
      <h1>Attack Tree — {tactic}</h1>
      <div class="legend">
        <span class="legend-item" style="background:{c_root}">Compromise Device (root)</span>
        <span class="legend-item" style="background:{c_tactic}">Tactic</span>
        <span class="legend-item" style="background:{c_cwe}">CWE</span>
        <span class="legend-item" style="background:{c_tech}">Technique</span>
        <span class="legend-item" style="background:{c_subtech}">Subtechnique</span>
        <span class="legend-item" style="background:{c_capec}">CAPEC</span>
        <span class="legend-item" style="background:{c_gate}">OR / AND gates</span>
      </div>
    </div>

    <div class="svg-scroll" id="svg-scroll">
      {svg_content}
    </div>

    <div class="graph-footer">
      Click any node to view its metadata in the right-hand panel. &nbsp;|&nbsp;
      Scroll / drag to navigate. &nbsp;|&nbsp;
      Press <kbd>Esc</kbd> to close the panel.
    </div>
  </div>

  <!-- ── Right column: metadata panel ── -->
  <div class="panel-col hidden" id="panel-col">
    <div class="panel-header">
      <h2 id="panel-title">Node metadata</h2>
      <button class="close-btn" id="close-btn" title="Close panel">&times;</button>
    </div>
    <div class="panel-body" id="panel-body">
      <div class="panel-placeholder">Click a node in the graph to see details here.</div>
    </div>
  </div>

</div>

<script>
// ── Data injected by Python ──────────────────────────────────────────────────
var NODE_HTML  = {node_html_json};
var NODES_DATA = {nodes_data_json};

// Build lookup: node_id -> {{label, type}}
var nodeInfo = {{}};
NODES_DATA.forEach(function(n) {{ nodeInfo[n.id] = n; }});

// ── DOM refs ─────────────────────────────────────────────────────────────────
var panelCol  = document.getElementById('panel-col');
var panelBody = document.getElementById('panel-body');
var panelTitle= document.getElementById('panel-title');
var closeBtn  = document.getElementById('close-btn');
var svgScroll = document.getElementById('svg-scroll');

var selectedGroup = null;

// ── Open panel with node data ─────────────────────────────────────────────────
function showPanel(nodeId) {{
  var info    = nodeInfo[nodeId] || {{}};
  var label   = info.label  || nodeId;
  var ntype   = info.type   || 'unknown';
  var htmlTbl = NODE_HTML[nodeId] || '<div class="meta-empty">No metadata recorded for this node.</div>';

  panelTitle.textContent = 'Node metadata';

  var badgeClass = 'badge-' + ntype;
  panelBody.innerHTML =
    '<p class="node-label-heading">' + escHtml(label) + '</p>' +
    '<span class="node-type-badge ' + badgeClass + '">' + escHtml(ntype) + '</span>' +
    htmlTbl;

  panelCol.classList.remove('hidden');
}}

function closePanel() {{
  panelCol.classList.add('hidden');
  if (selectedGroup) {{
    selectedGroup.classList.remove('selected-node');
    selectedGroup = null;
  }}
}}

function escHtml(s) {{
  return String(s)
    .replace(/&/g,'&amp;').replace(/</g,'&lt;')
    .replace(/>/g,'&gt;').replace(/"/g,'&quot;');
}}

// ── Wire up node clicks ───────────────────────────────────────────────────────
document.addEventListener('DOMContentLoaded', function() {{
  var groups = svgScroll.querySelectorAll('[data-node-id]');
  groups.forEach(function(g) {{
    g.addEventListener('click', function(ev) {{
      ev.stopPropagation();
      var nodeId = g.getAttribute('data-node-id');

      // Deselect previous
      if (selectedGroup && selectedGroup !== g) {{
        selectedGroup.classList.remove('selected-node');
      }}
      // Toggle if same node clicked twice
      if (selectedGroup === g) {{
        closePanel();
        return;
      }}
      selectedGroup = g;
      g.classList.add('selected-node');
      showPanel(nodeId);
    }});
  }});

  // Click empty canvas area → close panel
  svgScroll.addEventListener('click', function() {{ closePanel(); }});

  closeBtn.addEventListener('click', closePanel);

  document.addEventListener('keydown', function(e) {{
    if (e.key === 'Escape') closePanel();
  }});
}});
</script>
</body>
</html>
"""


def build_html(tactic: str, svg_str: str, node_html: dict, nodes_list: list) -> str:
    return _HTML_TEMPLATE.format(
        tactic        = html_lib.escape(tactic),
        svg_content   = svg_str,
        node_html_json  = json.dumps(node_html),
        nodes_data_json = json.dumps(nodes_list),
        c_root    = COLORS["root"],
        c_tactic  = COLORS["tactic"],
        c_cwe     = COLORS["cwe"],
        c_tech    = COLORS["tech"],
        c_subtech = COLORS["subtech"],
        c_capec   = COLORS["capec"],
        c_gate    = COLORS["gate"],
    )


# ─────────────────────────────────────────────────────────────────────────────
# NaN sanitiser
# ─────────────────────────────────────────────────────────────────────────────

def _clean(obj):
    if isinstance(obj, dict):
        return {k: _clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_clean(v) for v in obj]
    try:
        if pd.isna(obj):
            return "NaN"
    except Exception:
        pass
    return obj


# ─────────────────────────────────────────────────────────────────────────────
# Runner
# ─────────────────────────────────────────────────────────────────────────────

def generate_all():
    for i, tactic in enumerate(ordered_tactics):
        print(f"\n[{i+1}/{len(ordered_tactics)}] Building: {tactic}")

        dot, nodes_list, edges_list, node_html = build_tactic_graph(tactic, i)

        safe_name = tactic.replace(" ", "_")
        base      = os.path.join(OUTPUT_DIR,
                                 f"attack_tree_tactic_{i+1:02d}_{safe_name}")

        # 1. Save .dot source
        dot_path = base + ".dot"
        dot.save(dot_path)
        print(f"  DOT  → {dot_path}")

        # 2. Render SVG (cleanup keeps only .svg, removes intermediate)
        dot.render(base, format="svg", cleanup=True)
        svg_path = base + ".svg"
        print(f"  SVG  → {svg_path}")

        # 3. Post-process SVG: inject data-node-id attributes
        svg_str = inject_node_ids(svg_path)

        # 4. Build and save interactive HTML
        html_str  = build_html(tactic, svg_str, node_html, nodes_list)
        html_path = base + ".html"
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(html_str)
        print(f"  HTML → {html_path}")

        # 5. Save JSON metadata
        json_data = {
            "tactic": tactic,
            "nodes":  _clean(nodes_list),
            "edges":  _clean(edges_list),
        }
        json_path = base + ".json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4)
        print(f"  JSON → {json_path}")

        # 6. Optional PNG
        try:
            dot.render(base, format="png", cleanup=True)
            print(f"  PNG  → {base}.png")
        except Exception as e:
            print(f"  PNG  → skipped ({e})")

    print(f"\n✓ Done. Output folder: {OUTPUT_DIR}/")


if __name__ == "__main__":
    generate_all()

Found 9 tactics: ['Initial Access', 'Persistence', 'Privilege Escalation', 'Defense Evasion', 'Credential Access', 'Discovery', 'Lateral Movement', 'Collection', 'nan']

[1/9] Building: Initial Access
  DOT  → attack_tree_images/attack_tree_tactic_01_Initial_Access.dot
  SVG  → attack_tree_images/attack_tree_tactic_01_Initial_Access.svg
  HTML → attack_tree_images/attack_tree_tactic_01_Initial_Access.html
  JSON → attack_tree_images/attack_tree_tactic_01_Initial_Access.json
  PNG  → attack_tree_images/attack_tree_tactic_01_Initial_Access.png

[2/9] Building: Persistence
  DOT  → attack_tree_images/attack_tree_tactic_02_Persistence.dot
  SVG  → attack_tree_images/attack_tree_tactic_02_Persistence.svg
  HTML → attack_tree_images/attack_tree_tactic_02_Persistence.html
  JSON → attack_tree_images/attack_tree_tactic_02_Persistence.json
  PNG  → attack_tree_images/attack_tree_tactic_02_Persistence.png

[3/9] Building: Privilege Escalation
  DOT  → attack_tree_images/attack_tree_tactic_03_Pri

# HTML INTERACTIVE TREE (Full Tree)

In [ ]:

"""
comprehensive_attack_tree_builder.py (with path deduplication)

Generates:
 - comprehensive_attack_tree.svg  (Graphviz static layout w/ hover showing label only)
 - comprehensive_attack_tree.html (embeds the SVG; click nodes to see metadata in a right-side panel)
 - comprehensive_attack_tree.png  (optional, rasterized via Graphviz)

Strategy:
1. Build the full tree first (allowing CWEs to appear in multiple tactics)
2. Track all exploitation paths: (CWE, Technique, Subtechnique, CAPEC)
3. Prune duplicate paths that appear in later tactics
4. Keep only the first occurrence of each unique exploitation path

Place `device_compromise_dataset.xlsx` in the same folder or update DATASET_PATH.
"""
import os
import re
import json
import html as html_lib
import pandas as pd
from graphviz import Digraph
import xml.etree.ElementTree as ET
from collections import defaultdict

# -------------------------
# Config / file paths
# -------------------------
DATASET_PATH = "device_compromise_dataset.xlsx"
OUTPUT_DIR = "attack_tree_html_unified"
SVG_BASENAME = "comprehensive_attack_tree"
SVG_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".svg")
HTML_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".html")
PNG_OUTPUT = os.path.join(OUTPUT_DIR, SVG_BASENAME + ".png")

# MITRE tactic ordering (keeps consistent ordering)
MITRE_TACTIC_ORDER = [
    "Reconnaissance", "Resource Development", "Initial Access", "Execution", "Persistence",
    "Privilege Escalation", "Defense Evasion", "Credential Access", "Discovery",
    "Lateral Movement", "Collection", "Command and Control", "Exfiltration", "Impact"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------
# Read dataset
# -------------------------
df = pd.read_excel(DATASET_PATH, index_col=False)
df = df.fillna("")  # replace NaN with empty strings for simpler checks

if "tactics" not in df.columns:
    raise KeyError("Expected 'tactics' column in the dataset (case-sensitive).")

df["tactics"] = df["tactics"].astype(str).str.strip()

# Determine ordered tactics present in dataset
unique_tactics = [t for t in df["tactics"].unique() if t != ""]
ordered_tactics = [t for t in MITRE_TACTIC_ORDER if t in unique_tactics]
for t in unique_tactics:
    if t not in ordered_tactics:
        ordered_tactics.append(t)

print("Tactics (ordered):", ordered_tactics)

# -------------------------
# Helper functions
# -------------------------
def _normalize_name(s: str) -> str:
    """Normalize a column/key name: lowercase alphanum only to match flexible variants."""
    return re.sub(r'[^0-9a-z]', '', str(s).lower())

def extract_node_metadata(df_subset: pd.DataFrame, node_type: str):
    """
    Extract metadata for a node subset.

    Aggregate unique non-empty values across all rows for each key.
    Column matching is tolerant to case/spacing/underscore variations.
    """
    def build_cols_map(cols):
        return { _normalize_name(c): c for c in cols }

    cols_map = build_cols_map(df_subset.columns.tolist())

    if node_type == "cwe":
        keys = [
            'CVE ID', 'Description', 'cwe_Name', 'cwe_Abstraction', 'cwe_Structure', 'cwe_Status',
            'cwe_Description', 'cwe_Extended_Description', 'cwe_Related_Weaknesses', 'cwe_Weakness_Ordinality',
            'cwe_Language_Name', 'cwe_Technology_Class', 'cwe_Alternate_Terms', 'cwe_Alternate_Terms_With_Descriptions',
            'cwe_Background_Details', 'cwe_Modes_Of_Introduction', 'cwe_Likelihood_Of_Exploit',
            'cwe_Common_Consequences', 'cwe_Detection_Methods', 'cwe_Potential_Mitigations', 'cwe_Demonstrative_Examples',
            'cwe_Observed_Examples', 'attackVector', 'baseScore', 'baseSeverity', 'attackComplexity',
            'privilegesRequired', 'userInteraction', 'confidentialityImpact', 'availabilityImpact', 'exploitabilityScore'
        ]
    elif node_type == "capec":
        keys = [
            'CVE ID', 'capec_Name', 'capec_Mitre_Techniques', 'capec_Abstraction', 'capec_Status',
            'capec_Description', 'capec_Extended_Description', 'capec_Likelihood_Of_Attack', 'capec_Typical_Severity',
            'capec_Execution_Flow', 'capec_Prerequisites', 'capec_Skills_Required', 'capec_Resources_Required',
            'capec_Consequences', 'capec_Mitigations', 'capec_Example_Instances', 'capec_Related_Weaknesses',
            'capec_Taxonomy_Mappings'
        ]
    elif node_type == "tech":
        keys = [
            'CVE ID', 'Description', 'tech_description','technique_name', 'subtechnique_name', 'tactics', 'detection',
            'platforms', 'data sources', 'defenses bypassed', 'contributors', 'permissions required',
            'supports remote', 'system requirements', 'impact type', 'effective permissions',
            'attackVector', 'baseScore', 'baseSeverity', 'attackComplexity', 'privilegesRequired',
            'userInteraction', 'confidentialityImpact', 'availabilityImpact', 'exploitabilityScore'
        ]
    elif node_type == "subtech":
        keys = [
            'subtechnique_name', 'technique_name', 'tactics', 'detection', 'platforms', 'data sources'
        ]
    else:
        keys = df_subset.columns.tolist()

    meta = {}
    for k in keys:
        nk = _normalize_name(k)
        if nk in cols_map:
            colname = cols_map[nk]
        else:
            colname = None
        if colname and colname in df_subset.columns:
            vals = df_subset[colname].astype(str).map(str.strip)
            vals = [v for v in vals if v != "" and v.lower() != "nan"]
            uniq = []
            for v in vals:
                if v not in uniq:
                    uniq.append(v)
            if len(uniq) == 1:
                meta[k] = uniq[0]
            elif len(uniq) > 1:
                meta[k] = " | ".join(uniq)
    return meta

def format_metadata_for_html(metadata_dict, max_rows=60):
    """
    Convert metadata dict to an HTML table (string). Escapes values for safety.
    """
    if not metadata_dict:
        return "<div class='meta-empty'>(No metadata available)</div>"
    rows = []
    preferred_order = ["cwe_Name", "capec_Name", "technique_name", "subtechnique_name", "Description", "CVE ID"]
    added = set()
    for k in preferred_order:
        if k in metadata_dict:
            rows.append((k, metadata_dict[k])); added.add(k)
    for k, v in metadata_dict.items():
        if k in added: continue
        if len(rows) >= max_rows: break
        rows.append((k, v))
    table_html = ["<table class='meta-table' style='width:100%;border-collapse:collapse;'>"]
    for k, v in rows:
        k_esc = html_lib.escape(str(k))
        v_esc = html_lib.escape(str(v))
        table_html.append(
            f"<tr><th style='text-align:left;padding:8px;border-bottom:1px solid #eee;width:38%;vertical-align:top;background:#fafafa'>{k_esc}</th>"
            f"<td style='padding:8px;border-bottom:1px solid #eee;vertical-align:top'>{v_esc}</td></tr>"
        )
    table_html.append("</table>")
    return "\n".join(table_html)

def build_gate_tree(dot, children, gate_label, gate_prefix, gate_counter, nodes_data, edges_data, gate_color="#FFCCCB"):
    """
    Build a minimal binary tree of gate nodes representing a choice among `children`.
    Returns the root node id representing the OR (or the single child's id if only one child).
    Guarantees exactly one top-level gate for the group (no accidental double-wrapping).
    """
    if not children:
        return None
    if len(children) == 1:
        return children[0]
    def _build(sub_children):
        if len(sub_children) == 1:
            return sub_children[0]
        g_id = f"{gate_prefix}_{gate_counter[0]}"
        gate_counter[0] += 1
        dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
        nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
        mid = len(sub_children) // 2
        left = _build(sub_children[:mid])
        right = _build(sub_children[mid:])
        if left:
            dot.edge(g_id, left); edges_data.append({"from": g_id, "to": left})
        if right:
            dot.edge(g_id, right); edges_data.append({"from": g_id, "to": right})
        return g_id
    return _build(children)

def binary_gate(dot, parent, children, gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix):
    """
    Backwards-compatible parent->gate->children connector used in several places.
    """
    if len(children) == 0:
        return
    if len(children) == 1:
        dot.edge(parent, children[0]); edges_data.append({"from": parent, "to": children[0]}); return
    if len(children) == 2:
        g_id = f"{gate_prefix}_{gate_counter[0]}"; gate_counter[0] += 1
        dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
        nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
        dot.edge(parent, g_id); dot.edge(g_id, children[0]); dot.edge(g_id, children[1])
        edges_data.extend([{"from": parent, "to": g_id}, {"from": g_id, "to": children[0]}, {"from": g_id, "to": children[1]}])
        return
    mid = len(children) // 2
    g_id = f"{gate_prefix}_{gate_counter[0]}"; gate_counter[0] += 1
    dot.node(g_id, label=gate_label, shape="ellipse", fillcolor=gate_color, style="filled")
    nodes_data.append({"id": g_id, "label": gate_label, "type": "gate"})
    dot.edge(parent, g_id); edges_data.append({"from": parent, "to": g_id})
    binary_gate(dot, g_id, children[:mid], gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix)
    binary_gate(dot, g_id, children[mid:], gate_label, gate_color, gate_counter, nodes_data, edges_data, gate_prefix)

# -------------------------
# Build combined graph with deduplication
# -------------------------
def build_attack_tree():
    """
    Build attack tree with the following approach:
    1. First pass: collect all unique exploitation paths across all tactics
    2. Track first occurrence of each path (CWE, Technique, Subtechnique, CAPEC)
    3. Build tree structure maintaining only first occurrences
    """
    dot = Digraph(name="comprehensive_attack_tree", format="svg")
    dot.attr('graph', rankdir='TB', splines='ortho', nodesep='0.6', ranksep='1.0')
    dot.attr('node', fontname='Helvetica', fontsize='28')
    dot.attr('edge', fontsize='8')

    nodes_data = []
    edges_data = []
    node_metadata_map = {}

    # Root node
    root_id = "root"
    dot.node(root_id, label="Compromise Device", shape="box", fillcolor="#FFD700", style="filled")
    nodes_data.append({"id": root_id, "label": "Compromise Device", "type": "root", "metadata": {"Description": "Root node"}})

    # Tactic nodes
    tactic_ids = []
    for i, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{i}"
        dot.node(tid, label=tactic, shape="box", fillcolor="#98FB98", style="filled")
        nodes_data.append({"id": tid, "label": tactic, "type": "tactic", "metadata": {"tactic": tactic}})
        tactic_ids.append(tid)

    # Connect root -> tactics using binary SAND gates
    sand_counter = [0]
    binary_gate(dot, root_id, tactic_ids, "SAND", "#FFCCCB", sand_counter, nodes_data, edges_data, gate_prefix="sand")

    or_counter = [0]
    and_counter = [0]
    
    # -------------------------
    # PHASE 1: Discover all unique exploitation paths
    # -------------------------
    print("\n" + "="*80)
    print("PHASE 1: Discovering unique exploitation paths...")
    print("="*80)
    
    # Structure to track paths: path_key -> first_tactic_index
    path_first_occurrence = {}
    
    # Structure to organize: tactic_index -> cwe_name -> list of path_keys
    tactic_cwe_paths = defaultdict(lambda: defaultdict(list))
    
    for ti, tactic in enumerate(ordered_tactics):
        tactic_df = df[df["tactics"] == tactic]
        cwe_names = [x for x in tactic_df.get("cwe_Name", pd.Series([], dtype=object)).unique().tolist() if x != ""]
        
        for cwe in cwe_names:
            cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
            techniques = [x for x in cwe_subset.get("technique_name", pd.Series([], dtype=object)).unique().tolist() if x != ""]
            
            for tech in techniques:
                tech_subset = cwe_subset[cwe_subset["technique_name"] == tech]
                subtechs = [s for s in tech_subset.get("subtechnique_name", pd.Series([], dtype=object)).unique().tolist() if s != ""]
                
                if subtechs:
                    # Has subtechniques
                    for subtech in subtechs:
                        sub_rows = tech_subset[tech_subset["subtechnique_name"] == subtech]
                        capec_names = [c for c in sub_rows.get("capec_Name", pd.Series([], dtype=object)).unique().tolist() if c != ""]
                        
                        if not capec_names:
                            capec_names = ["CAPEC ?"]
                        
                        for cap in capec_names:
                            # Create path signature: (cwe, tech, subtech, capec)
                            path_key = (cwe, tech, subtech, cap)
                            
                            if path_key not in path_first_occurrence:
                                # First occurrence of this path
                                path_first_occurrence[path_key] = ti
                                tactic_cwe_paths[ti][cwe].append(path_key)
                                print(f"  [{tactic}] New path: {cwe} -> {tech} -> {subtech} -> {cap}")
                            else:
                                # Duplicate path - skip
                                first_tactic = ordered_tactics[path_first_occurrence[path_key]]
                                print(f"  [{tactic}] SKIP duplicate: {cwe} -> {tech} -> {subtech} -> {cap}")
                                print(f"           (Already in {first_tactic})")
                else:
                    # No subtechniques - use technique directly
                    capec_names = [c for c in tech_subset.get("capec_Name", pd.Series([], dtype=object)).unique().tolist() if c != ""]
                    
                    if not capec_names:
                        capec_names = ["CAPEC ?"]
                    
                    for cap in capec_names:
                        # Create path signature: (cwe, tech, "", capec)
                        path_key = (cwe, tech, "", cap)
                        
                        if path_key not in path_first_occurrence:
                            # First occurrence of this path
                            path_first_occurrence[path_key] = ti
                            tactic_cwe_paths[ti][cwe].append(path_key)
                            print(f"  [{tactic}] New path: {cwe} -> {tech} -> {cap}")
                        else:
                            # Duplicate path - skip
                            first_tactic = ordered_tactics[path_first_occurrence[path_key]]
                            print(f"  [{tactic}] SKIP duplicate: {cwe} -> {tech} -> {cap}")
                            print(f"           (Already in {first_tactic})")
    
    print(f"\nTotal unique exploitation paths: {len(path_first_occurrence)}")
    
    # -------------------------
    # PHASE 2: Build the tree structure with only unique paths
    # -------------------------
    print("\n" + "="*80)
    print("PHASE 2: Building attack tree structure...")
    print("="*80)
    
    # Track created nodes to reuse IDs
    path_to_and_gate = {}  # path_key -> and_gate_id
    
    for ti in sorted(tactic_cwe_paths.keys()):
        tactic = ordered_tactics[ti]
        tid = f"tactic_{ti}"
        
        print(f"\nBuilding tactic: {tactic}")
        
        cwe_node_ids = []
        
        for cwe, path_keys in tactic_cwe_paths[ti].items():
            if not path_keys:
                continue
            
            # Create CWE node for this tactic
            cwe_id = f"cwe_{ti}_{cwe.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')[:50]}"
            
            # Get metadata
            tactic_df = df[df["tactics"] == tactic]
            cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe]
            meta = extract_node_metadata(cwe_subset, "cwe")
            
            dot.node(cwe_id, label=cwe, shape="note", fillcolor="#ADD8E6", style="filled")
            nodes_data.append({"id": cwe_id, "label": cwe, "type": "cwe", "metadata": meta})
            node_metadata_map[cwe_id] = format_metadata_for_html(meta)
            cwe_node_ids.append(cwe_id)
            
            print(f"  CWE: {cwe} ({len(path_keys)} paths)")
            
            # Create exploitation paths for this CWE
            and_gates_for_cwe = []
            
            for path_key in path_keys:
                cwe_name, tech, subtech, cap = path_key
                
                # Create AND gate for this path
                and_id = f"and_{ti}_{and_counter[0]}"
                and_counter[0] += 1
                dot.node(and_id, label="AND", shape="ellipse", fillcolor="#FFCCCB", style="filled")
                nodes_data.append({"id": and_id, "label": "AND", "type": "gate"})
                and_gates_for_cwe.append(and_id)
                path_to_and_gate[path_key] = and_id
                
                # Get the data for this specific path
                tactic_df = df[df["tactics"] == tactic]
                cwe_subset = tactic_df[tactic_df["cwe_Name"] == cwe_name]
                tech_subset = cwe_subset[cwe_subset["technique_name"] == tech]
                
                if subtech:
                    # Has subtechnique
                    sub_rows = tech_subset[tech_subset["subtechnique_name"] == subtech]
                    
                    # Create subtechnique node
                    sub_id = f"subtech_{ti}_{and_counter[0]}"
                    sub_meta = extract_node_metadata(sub_rows, "subtech")
                    dot.node(sub_id, label=subtech, shape="box", fillcolor="#FFFACD", style="filled")
                    nodes_data.append({"id": sub_id, "label": subtech, "type": "subtech", "metadata": sub_meta})
                    node_metadata_map[sub_id] = format_metadata_for_html(sub_meta)
                    
                    # Connect AND -> subtechnique
                    dot.edge(and_id, sub_id)
                    edges_data.append({"from": and_id, "to": sub_id})
                    
                    # Create CAPEC node
                    cap_id = f"capec_{ti}_{and_counter[0]}"
                    if cap != "CAPEC ?":
                        cap_rows = sub_rows[sub_rows["capec_Name"] == cap]
                        cap_meta = extract_node_metadata(cap_rows, "capec")
                    else:
                        cap_rows = sub_rows
                        cap_meta = {}
                    
                    dot.node(cap_id, label=cap, shape="box", fillcolor="#FFB6C1", style="filled")
                    nodes_data.append({"id": cap_id, "label": cap, "type": "capec", "metadata": cap_meta})
                    node_metadata_map[cap_id] = format_metadata_for_html(cap_meta)
                    
                    # Connect AND -> CAPEC
                    dot.edge(and_id, cap_id)
                    edges_data.append({"from": and_id, "to": cap_id})
                    
                    print(f"    Path: {tech} -> {subtech} -> {cap}")
                else:
                    # No subtechnique - use technique
                    tech_id = f"tech_{ti}_{and_counter[0]}"
                    tech_meta = extract_node_metadata(tech_subset, "tech")
                    dot.node(tech_id, label=tech, shape="box", fillcolor="#90EE90", style="filled")
                    nodes_data.append({"id": tech_id, "label": tech, "type": "tech", "metadata": tech_meta})
                    node_metadata_map[tech_id] = format_metadata_for_html(tech_meta)
                    
                    # Connect AND -> technique
                    dot.edge(and_id, tech_id)
                    edges_data.append({"from": and_id, "to": tech_id})
                    
                    # Create CAPEC node
                    cap_id = f"capec_{ti}_{and_counter[0]}"
                    if cap != "CAPEC ?":
                        cap_rows = tech_subset[tech_subset["capec_Name"] == cap]
                        cap_meta = extract_node_metadata(cap_rows, "capec")
                    else:
                        cap_rows = tech_subset
                        cap_meta = {}
                    
                    dot.node(cap_id, label=cap, shape="box", fillcolor="#FFB6C1", style="filled")
                    nodes_data.append({"id": cap_id, "label": cap, "type": "capec", "metadata": cap_meta})
                    node_metadata_map[cap_id] = format_metadata_for_html(cap_meta)
                    
                    # Connect AND -> CAPEC
                    dot.edge(and_id, cap_id)
                    edges_data.append({"from": and_id, "to": cap_id})
                    
                    print(f"    Path: {tech} -> {cap}")
            
            # Connect CWE to its exploitation paths
            if len(and_gates_for_cwe) == 1:
                # Direct connection
                dot.edge(cwe_id, and_gates_for_cwe[0])
                edges_data.append({"from": cwe_id, "to": and_gates_for_cwe[0]})
            else:
                # OR gate for multiple paths
                or_root = build_gate_tree(dot, and_gates_for_cwe, "OR", "or_exp", or_counter, nodes_data, edges_data)
                if or_root:
                    dot.edge(cwe_id, or_root)
                    edges_data.append({"from": cwe_id, "to": or_root})
        
        # Connect tactic to its CWEs
        if cwe_node_ids:
            binary_gate(dot, tid, cwe_node_ids, "OR", "#FFCCCB", or_counter, nodes_data, edges_data, gate_prefix="or")

    # -------------------------
    # Post-build pass: compute tactic reachability (CWEs, techniques, CAPECs)
    # -------------------------
    print("\n" + "="*80)
    print("PHASE 3: Computing tactic metadata...")
    print("="*80)
    
    adj = {}
    for e in edges_data:
        u = e.get("from"); v = e.get("to")
        if u is None or v is None: continue
        adj.setdefault(u, []).append(v)

    id_to_type = { nd["id"]: nd.get("type", "") for nd in nodes_data }
    id_to_label = { nd["id"]: nd.get("label", nd["id"]) for nd in nodes_data }

    def reachable_nodes(start_id):
        seen = set()
        q = [start_id]
        seen.add(start_id)
        result = []
        while q:
            cur = q.pop(0)
            for nb in adj.get(cur, []):
                if nb not in seen:
                    seen.add(nb)
                    result.append(nb)
                    q.append(nb)
        return result

    for i, tactic in enumerate(ordered_tactics):
        tid = f"tactic_{i}"
        reached = reachable_nodes(tid)
        cwe_labels = []
        tech_labels = []
        capec_labels = []
        for nid in reached:
            ntype = id_to_type.get(nid, "")
            nlabel = id_to_label.get(nid, nid)
            if ntype == "cwe":
                if nlabel not in cwe_labels: cwe_labels.append(nlabel)
            elif ntype in ("tech", "subtech"):
                if nlabel not in tech_labels: tech_labels.append(nlabel)
            elif ntype == "capec":
                if nlabel not in capec_labels: capec_labels.append(nlabel)
        tactic_meta = {
            "tactic": tactic,
            "cwes": ", ".join(cwe_labels) if cwe_labels else "",
            "techniques": ", ".join(tech_labels) if tech_labels else "",
            "capecs": ", ".join(capec_labels) if capec_labels else ""
        }
        for nd in nodes_data:
            if nd["id"] == tid:
                nd_meta = nd.get("metadata", {}).copy() if isinstance(nd.get("metadata", {}), dict) else {}
                nd_meta.update(tactic_meta)
                nd["metadata"] = nd_meta
                break
        node_metadata_map[tid] = format_metadata_for_html(tactic_meta)

    print(f"\nTree construction complete!")
    print(f"Total nodes: {len(nodes_data)}")
    print(f"Total edges: {len(edges_data)}")
    return dot, nodes_data, edges_data, node_metadata_map

# -------------------------
# SVG post-processing: inject attributes (data-node-id), set harmless title and collect positions
# -------------------------
def inject_tooltips_and_extract_positions(svg_path, node_metadata_map):
    ns = {'svg': 'http://www.w3.org/2000/svg'}
    ET.register_namespace("", ns['svg'])
    tree = ET.parse(svg_path)
    root = tree.getroot()

    node_positions = {}

    for g in root.findall(".//svg:g", ns):
        title = g.find("svg:title", ns)
        if title is None:
            continue
        original_title_text = title.text if title.text is not None else ""
        node_id = original_title_text

        # Find text label inside group (fallback)
        tt = g.find(".//svg:text", ns)
        label_text = tt.text if tt is not None and tt.text is not None else node_id

        # Set <title> to label only (keeps hover small)
        title.text = label_text

        # Tag group with data-node-id attribute
        if node_id:
            g.set("data-node-id", node_id)

        # extract translate(x,y) coordinates when present
        transform = g.get("transform", "")
        m = re.search(r"translate\(\s*([0-9\.\-]+)[,\s]+([0-9\.\-]+)\s*\)", transform)
        if m:
            try:
                x = float(m.group(1)); y = float(m.group(2))
                node_positions[node_id] = {"x": x, "y": y}
            except Exception:
                pass

    tree.write(svg_path, encoding="utf-8", xml_declaration=True)
    return node_positions

# -------------------------
# HTML wrapper creation (larger display) with right-side panel metadata JS
# -------------------------
def save_html_with_svg(svg_path, html_path, node_positions, nodes_data, node_metadata_map):
    with open(svg_path, "r", encoding="utf-8") as f:
        svg_content = f.read()

    # node_metadata_map values already contain HTML (created by format_metadata_for_html)
    meta_json = json.dumps(node_metadata_map)
    positions_json = json.dumps(node_positions)
    nodes_data_json = json.dumps(nodes_data)

    # We now create a two-column flex layout: .main (left, expanding) and #panel-root (right, fixed width)
    script_blob = f"""
<script>
window.ATTACK_TREE_META = {{
  "node_positions": {positions_json},
  "short_metadata": {meta_json},
  "nodes_data": {nodes_data_json}
}};

document.addEventListener('DOMContentLoaded', function() {{
  // find panel root (placeholder)
  let panelRoot = document.getElementById('panel-root');
  if (!panelRoot) {{
    // fallback: create one at end of body
    panelRoot = document.createElement('div');
    panelRoot.id = 'panel-root';
    document.body.appendChild(panelRoot);
  }}

  // create panel element inside panelRoot (so layout is two-column)
  let panel = document.createElement('div');
  panel.id = 'metadata-panel';
  panel.style.boxSizing = 'border-box';
  panel.style.height = '100%';
  panel.style.width = '100%';
  panel.style.background = '#fff';
  panel.style.overflow = 'auto';
  panel.style.padding = '14px';
  panel.style.fontFamily = 'Arial, Helvetica, sans-serif';
  panel.style.fontSize = '15px';
  panel.style.display = 'none';
  panelRoot.appendChild(panel);

  // header with close button
  let header = document.createElement('div');
  header.style.display = 'flex';
  header.style.alignItems = 'center';
  header.style.justifyContent = 'space-between';
  header.style.marginBottom = '10px';

  let title = document.createElement('div');
  title.innerText = 'Node metadata';
  title.style.fontWeight = '700';
  title.style.fontSize = '18px';
  header.appendChild(title);

  let closeBtn = document.createElement('button');
  closeBtn.innerHTML = '&times;';
  closeBtn.title = 'Close';
  closeBtn.style.border = 'none';
  closeBtn.style.background = 'transparent';
  closeBtn.style.fontSize = '24px';
  closeBtn.style.cursor = 'pointer';
  closeBtn.style.lineHeight = '1';
  closeBtn.style.padding = '0 6px';
  closeBtn.addEventListener('click', function() {{ panel.style.display = 'none'; }});
  header.appendChild(closeBtn);

  panel.appendChild(header);

  let content = document.createElement('div');
  content.id = 'metadata-panel-content';
  panel.appendChild(content);

  let foot = document.createElement('div');
  foot.style.marginTop = '12px';
  foot.style.fontSize = '13px';
  foot.style.color = '#666';
  foot.innerText = 'Click a node in the graph to view metadata here. Press × to close.';
  panel.appendChild(foot);

  // Attach click listeners to any svg group that has data-node-id
  let svgWrap = document.querySelector('.svg-wrap');
  if (!svgWrap) return;
  let nodeGroups = svgWrap.querySelectorAll('[data-node-id]');
  nodeGroups.forEach(function(g) {{
    g.style.cursor = 'pointer';
    g.addEventListener('click', function(ev) {{
      ev.stopPropagation();
      let nodeId = g.getAttribute('data-node-id');
      let html = window.ATTACK_TREE_META.short_metadata[nodeId] || "";
      if (!html || html === "") {{
        let found = (window.ATTACK_TREE_META.nodes_data || []).find(x => x.id === nodeId);
        if (found && found.label) {{
          html = '<div style="font-weight:600;margin-bottom:8px">' + found.label + '</div>';
        }} else {{
          html = '<div class="meta-empty">(No metadata)</div>';
        }}
      }}
      let subtitle = '<div style="color:#555;font-size:13px;margin-bottom:10px">ID: ' + nodeId + '</div>';
      content.innerHTML = subtitle + html;
      panel.style.display = 'block';
      panel.scrollTop = 0;
    }});
  }});

  // Optional: close panel with Escape
  document.addEventListener('keydown', function(e) {{
    if (e.key === 'Escape') panel.style.display = 'none';
  }});
}});
</script>
"""

    # CSS & layout: use flex to let svg-wrap expand and occupy remaining screen space
    html_doc = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Comprehensive Attack Tree (Deduplicated)</title>
  <style>
    html, body {{ height: 100%; margin: 0; padding: 0; }}
    body {{ font-family: Arial, Helvetica, sans-serif; background: #fafafa; font-size:16px; }}
    .app {{ display:flex; height: calc(100vh - 20px); gap: 12px; padding: 10px; box-sizing: border-box; }}
    .main {{ flex: 1 1 auto; display:flex; flex-direction: column; min-width: 200px; }}
    .topbar {{ padding-bottom: 8px; }}
    h1 {{ font-size: 24px; margin: 0 0 6px 0; }}
    .legend {{ margin-top: 6px; font-size: 15px; }}
    .legend span {{ display: inline-block; margin-right: 12px; padding: 4px 8px; border-radius:4px; font-size:14px; }}
    p.note {{ font-size: 15px; color: #333; margin-top:8px; }}
    /* svg-wrap expands to fill available vertical space */
    .svg-wrap {{ flex: 1 1 auto; border: 1px solid #ddd; padding: 6px; background: #fff; overflow: auto; min-height: 200px; }}
    /* Let the inline SVG scale to the container height */
    .svg-wrap svg {{ height: 100%; width: auto; display: block; }}
    /* Panel root (right column) is fixed width but responsive on small screens */
    #panel-root {{ flex: 0 0 420px; max-width: 42vw; min-width: 320px; display:flex; flex-direction:column; box-sizing: border-box; }}
    @media (max-width: 1000px) {{
      #panel-root {{ min-width: 260px; max-width: 40vw; }}
      .legend span {{ font-size:13px; }}
    }}
    /* small metadata table styles (panel uses larger font) */
    .meta-table th {{ background:#fafafa; font-size:14px; }}
    .meta-table td {{ font-size:14px; }}
    .meta-empty {{ color:#666; font-style:italic; }}
  </style>
</head>
<body>
  <div class="app">
    <div class="main">
      <div class="topbar">
        <h1>Comprehensive Attack Tree (Deduplicated Paths)</h1>
        <div class="legend">
          <span style="background:#FFD700">Root</span>
          <span style="background:#98FB98">Tactic</span>
          <span style="background:#ADD8E6">CWE</span>
          <span style="background:#90EE90">Technique</span>
          <span style="background:#FFFACD">Subtechnique</span>
          <span style="background:#FFB6C1">CAPEC</span>
          <span style="background:#FFCCCB">Gates</span>
        </div>
      </div>

      <div class="svg-wrap">
        {svg_content}
      </div>

      <p class="note">Click a node to view metadata in the right-side panel. Identical exploitation paths appear only once at their first occurrence.</p>
    </div>

    <!-- placeholder for metadata panel (right column) -->
    <div id="panel-root" aria-hidden="false">
      <!-- JS will inject the metadata panel UI here -->
    </div>
  </div>

  {script_blob}
</body>
</html>
"""
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_doc)

# -------------------------
# Main run
# -------------------------
def main():
    print("="*80)
    print("ATTACK TREE BUILDER WITH PATH DEDUPLICATION")
    print("="*80)
    print(f"Dataset: {DATASET_PATH}")
    print(f"Output directory: {OUTPUT_DIR}")
    
    print("\nBuilding graph...")
    dot, nodes_data, edges_data, node_metadata_map = build_attack_tree()

    # Render to SVG using Graphviz. dot.render expects filename without extension.
    basepath = os.path.join(OUTPUT_DIR, SVG_BASENAME)
    print("\n" + "="*80)
    print("Rendering SVG (Graphviz). Ensure 'dot' binary available on PATH...")
    print("="*80)
    dot.render(basepath, format="svg", cleanup=True)
    svg_path = basepath + ".svg"
    print("SVG rendered at:", svg_path)

    # Inject attributes (data-node-id) and extract node positions
    print("\nInjecting attributes into SVG and extracting node positions...")
    node_positions = inject_tooltips_and_extract_positions(svg_path, node_metadata_map)
    print(f"Extracted positions for {len(node_positions)} nodes.")

    # Save HTML wrapper (with large canvas)
    print("\nSaving HTML wrapper (large canvas)...")
    save_html_with_svg(svg_path, HTML_OUTPUT, node_positions, nodes_data, node_metadata_map)
    print("HTML saved at:", HTML_OUTPUT)

    # Try PNG render for convenience
    try:
        print("\nRendering PNG for convenience...")
        dot.render(basepath, format="png", cleanup=True)
        print("PNG saved at:", basepath + ".png")
    except Exception as e:
        print("PNG rendering failed (optional). Error:", e)

    print("\n" + "="*80)
    print("COMPLETE!")
    print("="*80)
    print("Outputs:")
    print(" - SVG:", svg_path)
    print(" - HTML:", HTML_OUTPUT)
    print(" - PNG (optional):", basepath + ".png")
    print("="*80)

if __name__ == "__main__":
    main()

Tactics (ordered): ['Initial Access', 'Persistence', 'Privilege Escalation', 'Defense Evasion', 'Credential Access', 'Discovery', 'Lateral Movement', 'Collection']
ATTACK TREE BUILDER WITH PATH DEDUPLICATION
Dataset: device_compromise_dataset.xlsx
Output directory: attack_tree_html_unified

Building graph...

PHASE 1: Discovering unique exploitation paths...
  [Initial Access] New path: Use of Hard-coded Credentials -> Valid Accounts -> Valid Accounts: Default Accounts -> Try Common or Default Usernames and Passwords
  [Persistence] New path: Incorrect Permission Assignment for Critical Resource -> Hijack Execution Flow -> Hijack Execution Flow: Services File Permissions Weakness -> Accessing Functionality Not Properly Constrained by ACLs
  [Persistence] New path: Incorrect Permission Assignment for Critical Resource -> Hijack Execution Flow -> Hijack Execution Flow: Services File Permissions Weakness -> Using Malicious Files
  [Persistence] New path: Incorrect Permission Assignment fo